# Atelier 2.B — Découpage et recherche hybride

**Durée : 30 min**

Quelle stratégie de découpage et quel mode de recherche retrouvent le mieux les bons extraits ? On ne devine pas : on **mesure**, sur le golden set de 32 questions (`donnees/golden_set.json`).

Cet atelier n'appelle aucun LLM : tout tourne en local, sans quota.

In [ ]:
import pandas as pd
import rag

documents = rag.charger_corpus()
encodeur = rag.Encodeur()
QUESTIONS = [q for q in rag.charger_golden_set() if q["indices"]]  # questions dont la réponse est dans le corpus
K = 4                                                                # extraits transmis au modèle
print(len(QUESTIONS), "questions évaluées")

## Les métriques

Pour chaque question, le golden set donne des **indices** : des passages qui doivent figurer dans les extraits retrouvés pour qu'on puisse répondre (par exemple « 8 heures ouvrées »).

- **Couverture** (rappel du contexte) : part des indices présents dans les `K` extraits retrouvés, en moyenne sur les questions.
- **Context precision@k** : les extraits pertinents (qui contiennent un indice) sont-ils **en tête** ? Moyenne des precision@i aux rangs pertinents, comme dans RAGAS : 1 si tous les extraits pertinents sont en tête.
- **MRR** (*mean reciprocal rank*) : 1 si le premier extrait est pertinent, 1/2 si c'est le deuxième, etc.

L'atelier 2.C calcule les mêmes métriques avec un LLM juge (bibliothèque RAGAS).

In [ ]:
def mesurer(index, mode="dense", k=K):
    couvertures, precisions, inverses_rang = [], [], []
    for q in QUESTIONS:
        trouves = index.rechercher(q["question"], k, mode)
        couvertures.append(rag.couverture(trouves, q["indices"]))
        precisions.append(rag.precision_moyenne(trouves, q["indices"]))
        rang = rag.rang_premier_pertinent(trouves, q["indices"])
        inverses_rang.append(1 / rang if rang else 0)
    n = len(QUESTIONS)
    return {"couverture": round(sum(couvertures) / n, 3),
            "context_precision": round(sum(precisions) / n, 3),
            "mrr": round(sum(inverses_rang) / n, 3)}

## 1. Stratégies de découpage

On compare la taille des extraits, le chevauchement, le découpage par sections et l'ajout du titre du document en tête d'extrait. La colonne `caracteres_transmis` estime ce que coûte le contexte envoyé au modèle (`K` × taille moyenne).

In [ ]:
CONFIGURATIONS = {
    "fixe 800": dict(strategie="fixe", taille=800, chevauchement=0),
    "récursif 300": dict(strategie="recursif", taille=300, chevauchement=50),
    "récursif 800": dict(strategie="recursif", taille=800, chevauchement=100),
    "récursif 1500": dict(strategie="recursif", taille=1500, chevauchement=150),
    "sections 800": dict(strategie="sections", taille=800, chevauchement=100),
    "sections 800 + entête": dict(strategie="sections", taille=800, chevauchement=100, entete=True),
}

lignes, index_par_config = [], {}
for nom, config in CONFIGURATIONS.items():
    extraits = rag.decouper(documents, **config)
    index = rag.Index(extraits, encodeur)
    index_par_config[nom] = index
    taille = sum(len(e.texte) for e in extraits) / len(extraits)
    lignes.append({"configuration": nom, "extraits": len(extraits), "taille_moyenne": round(taille),
                   "caracteres_transmis": round(K * taille), **mesurer(index)})

pd.DataFrame(lignes).set_index("configuration")

**Questions**
1. Les grands extraits obtiennent-ils une meilleure couverture ? À quel prix en contexte transmis ?
2. Que change l'en-tête (titre du document) ajouté à chaque extrait ?
3. Le découpage fixe coupe au milieu des phrases : cela se voit-il dans les mesures ?

## 2. Les questions qui échouent

Pour la configuration retenue, quelles questions n'ont pas tous leurs indices dans les extraits retrouvés ?

In [ ]:
CONFIG = "sections 800 + entête"
index = index_par_config[CONFIG]

for q in QUESTIONS:
    trouves = index.dense(q["question"], K)
    if rag.couverture(trouves, q["indices"]) < 1:
        print(f"{q['id']} [{q['type']}] {q['question']}")
        print("   attendu dans :", q["sources"], "| indices :", q["indices"])
        print("   retrouvé     :", [e.source for e in trouves], "\n")

## 3. Recherche dense, lexicale, hybride

- **Dense** : similarité des embeddings ; comprend les reformulations et les synonymes.
- **Lexicale** (BM25) : mots communs, pondérés par leur rareté ; excelle sur les références exactes (« EN 1335 », « NS-4 »).
- **Hybride** : fusion des deux classements par *Reciprocal Rank Fusion* : chaque extrait reçoit la somme de 1 / (60 + rang) sur les deux classements.

In [ ]:
modes = {mode: mesurer(index, mode) for mode in ("dense", "lexical", "hybride")}
pd.DataFrame(modes).T

In [ ]:
def couverture_par_type(mode):
    resultats = {}
    for q in QUESTIONS:
        resultats.setdefault(q["type"], []).append(rag.couverture(index.rechercher(q["question"], K, mode), q["indices"]))
    return {t: round(sum(v) / len(v), 2) for t, v in resultats.items()}

pd.DataFrame({mode: couverture_par_type(mode) for mode in ("dense", "lexical", "hybride")})

## 4. Là où les modes divergent

In [ ]:
for q in QUESTIONS:
    dense = rag.couverture(index.dense(q["question"], K), q["indices"])
    lexical = rag.couverture(index.lexical(q["question"], K), q["indices"])
    if dense != lexical:
        gagnant = "dense" if dense > lexical else "lexical"
        print(f"{gagnant:8} l'emporte | {q['id']} {q['question']}  (dense {dense:.2f}, lexical {lexical:.2f})")

✏️ Ajoutez au jeu de référence deux questions qui contiennent une référence produit exacte (par exemple « Le DK-200 gère-t-il deux écrans 4K ? ») et deux questions formulées sans aucun mot du corpus. Quel mode les retrouve ?

## À retenir

- Le découpage se **mesure** : taille, chevauchement, sections et en-têtes changent la couverture, et le coût du contexte transmis.
- Le dense comprend le sens, le lexical les références exactes. L'hybride n'est pas automatiquement meilleur : il dépend du corpus et des questions.
- Un jeu de questions de référence, avec ce qu'il faut retrouver, est l'outil de base de toute amélioration.